# 03 — Separar y filtrar autores UNAM

Este notebook transforma la unión canónica en una estructura **publicación + autor UNAM**.

Principios de esta fase:

- conserva las 15 columnas canónicas;
- no crea `Base_origen`;
- no deduplica ni fusiona publicaciones;
- no completa DOI/ISBN/ISSN/Abstract/Keywords;
- no reclasifica `Area`;
- separa autor–afiliación según la estructura real de cada fuente;
- usa otras representaciones del mismo artículo y el archivo de la tutora solo como evidencia contextual;
- los registros con **más de 10 autores, afiliaciones o referencias**, los casos aún ambiguos y los autores con más de dos afiliaciones pasan a **un único archivo de revisión manual**;
- el CSV final solo se genera cuando existe `casos_revision_resueltos.csv`.

La comparación flexible de nombres se restringe a publicaciones ya identificadas por DOI o por título+año; no se utiliza fuzzy matching global.

## Importaciones

In [ ]:
import os
import re
import html
import hashlib
import unicodedata
from collections import defaultdict

import pandas as pd

## 1. RUTAS Y COLUMNAS

In [ ]:
archivo = "../02_modelo_canonico/03_union/Canonico_Union_Trabajo.csv"

archivo_tutora_xlsx = "../00_control/UNAM_Completo_Corregido.xlsx"
archivo_tutora_csv = "../00_control/UNAM_Completo_Corregido.csv"

carpeta_salida = "../04_Limpieza/01_Internos_unam"
os.makedirs(carpeta_salida, exist_ok=True)

salida_automatica = f"{carpeta_salida}/autores_unam_separados_automatico.csv"
revision_manual = f"{carpeta_salida}/casos_revision_manual.csv"

# Este archivo se llena manualmente después de revisar casos_revision_manual.csv.
revision_resuelta = f"{carpeta_salida}/casos_revision_resueltos.csv"
salida_final = f"{carpeta_salida}/autores_unam_separados.csv"

columnas = [
    "Fuente_origen", "indice", "Titulo", "Año", "Autor_norm",
    "Afiliacion1", "Afiliacion2", "ISBN", "ISSN", "Doi",
    "URL", "Area", "SubArea", "Keywords", "Abstract"
]

## 2. FUNCIONES GENERALES

In [ ]:
def sha256(ruta):
    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        for bloque in iter(lambda: f.read(1024 * 1024), b""):
            h.update(bloque)
    return h.hexdigest()


def decodificar_html(texto, max_iter=10):
    """Decodifica HTML antes de separar por ';'."""
    texto = "" if texto is None else str(texto)

    for _ in range(max_iter):
        nuevo = html.unescape(texto)
        if nuevo == texto:
            break
        texto = nuevo

    return unicodedata.normalize("NFC", texto).strip()


def separar_punto_coma(texto):
    texto = decodificar_html(texto)
    return [x.strip() for x in texto.split(";") if x.strip()]


def sin_acentos(texto):
    return "".join(
        c for c in unicodedata.normalize("NFKD", texto)
        if not unicodedata.combining(c)
    )


def normalizar_texto(texto):
    """Solo para comparar; no reemplaza el valor guardado."""
    texto = sin_acentos(decodificar_html(texto)).lower()
    texto = re.sub(r"[^a-z0-9]+", " ", texto)
    return " ".join(texto.split())


def normalizar_doi(doi):
    doi = decodificar_html(doi).lower().strip()
    doi = re.sub(r"^https?://(?:dx\.)?doi\.org/", "", doi)
    doi = re.sub(r"^doi:\s*", "", doi)
    return re.sub(r"\s+", "", doi)


def normalizar_titulo(titulo):
    return normalizar_texto(titulo)


def unicos_no_vacios(valores):
    resultado = []
    vistos = set()

    for valor in valores:
        valor = decodificar_html(valor)
        clave = normalizar_texto(valor)

        if valor and clave and clave not in vistos:
            resultado.append(valor)
            vistos.add(clave)

    return resultado

## 3. REFERENCIAS NUMÉRICAS Y COMPARACIÓN DE NOMBRES

In [ ]:
# Scopus Author ID: (57191896522)
SCOPUS_ID_RE = re.compile(r"\s*\((\d{7,12})\)\s*$")

# Referencias institucionales: (1), (1,2), (1, 2, 3)
REF_AUTOR_RE = re.compile(r"\s*\((\d{1,3}(?:\s*,\s*\d{1,3})*)\)\s*$")

# Catálogo: (1) institución ... (2) institución ...
MARCADOR_AFILIACION_RE = re.compile(r"(?:(?<=^)|(?<=\s))\((\d{1,3})\)\s*")


def quitar_scopus_id(autor):
    return SCOPUS_ID_RE.sub("", decodificar_html(autor)).strip()


def referencias_autor(autor):
    m = REF_AUTOR_RE.search(decodificar_html(autor))
    if not m:
        return []
    return [x.strip() for x in m.group(1).split(",") if x.strip()]


def quitar_referencias_autor(autor):
    return REF_AUTOR_RE.sub("", decodificar_html(autor)).strip()


def mapa_afiliaciones_numeradas(afiliacion1, afiliacion2=""):
    """
    Construye {numero: institucion}.
    La separación ocurre por el siguiente marcador (n), no por ';'.
    """
    texto = " ".join(
        x for x in [
            decodificar_html(afiliacion1),
            decodificar_html(afiliacion2)
        ]
        if x
    ).strip()

    encontrados = list(MARCADOR_AFILIACION_RE.finditer(texto))
    resultado = {}

    for i, m in enumerate(encontrados):
        fin = encontrados[i + 1].start() if i + 1 < len(encontrados) else len(texto)
        afiliacion = texto[m.end():fin].strip(" ;")

        if afiliacion:
            resultado[m.group(1)] = afiliacion

    return resultado


def expandir_umlaut(texto):
    reemplazos = {
        "ä": "ae", "ö": "oe", "ü": "ue",
        "Ä": "Ae", "Ö": "Oe", "Ü": "Ue", "ß": "ss"
    }

    for a, b in reemplazos.items():
        texto = texto.replace(a, b)

    return sin_acentos(texto)


def tokens_nombre(nombre, modo="acentos"):
    nombre = decodificar_html(nombre)
    nombre = SCOPUS_ID_RE.sub("", nombre)
    nombre = REF_AUTOR_RE.sub("", nombre).strip()

    if "," in nombre:
        apellido, nombres = [x.strip() for x in nombre.split(",", 1)]
        partes = re.findall(r"[A-Za-zÀ-ÖØ-öø-ÿ]+", nombres)
        desarrolladas = []

        for parte in partes:
            if parte.isupper() and 2 <= len(parte) <= 4:
                desarrolladas.extend(list(parte))
            else:
                desarrolladas.append(parte)

        nombre = " ".join(desarrolladas + [apellido])

    nombre = expandir_umlaut(nombre) if modo == "umlaut" else sin_acentos(nombre)
    return [x.lower() for x in re.findall(r"[A-Za-z0-9]+", nombre) if x]


def token_compatible(a, b):
    return (
        a == b
        or (len(a) == 1 and b.startswith(a))
        or (len(b) == 1 and a.startswith(b))
    )


def comparar_tokens(a, b):
    if not a or not b:
        return False

    if a == b:
        return True

    if len(a) == len(b) and all(token_compatible(x, y) for x, y in zip(a, b)):
        return True

    corto, largo = (a, b) if len(a) < len(b) else (b, a)

    if (
        len(corto) >= 2
        and token_compatible(corto[0], largo[0])
        and token_compatible(corto[-1], largo[-1])
    ):
        j = 0

        for token in corto:
            encontrado = False

            while j < len(largo):
                if token_compatible(token, largo[j]):
                    encontrado = True
                    j += 1
                    break
                j += 1

            if not encontrado:
                return False

        return True

    return False


def nombres_compatibles(a, b):
    """
    Se usa solo dentro de una publicación ya identificada por DOI
    o por título+año. No es fuzzy matching global.
    """
    for modo in ("acentos", "umlaut"):
        if comparar_tokens(tokens_nombre(a, modo), tokens_nombre(b, modo)):
            return True
    return False

## 4. SCOPUS, WOS Y UNAM

In [ ]:
def afiliaciones_scopus(autor, entrada_autor, catalogo):
    """
    Afiliacion2 indica qué instituciones corresponden al autor.
    Afiliacion1 sirve como catálogo de nombres institucionales completos.
    """
    entrada = decodificar_html(entrada_autor)
    entrada_norm = normalizar_texto(entrada)

    encontradas = [
        institucion
        for institucion in catalogo
        if normalizar_texto(institucion)
        and normalizar_texto(institucion) in entrada_norm
    ]
    encontradas = unicos_no_vacios(encontradas)

    if not encontradas:
        return [], False

    posiciones = [
        (entrada.find(inst), inst)
        for inst in encontradas
        if entrada.find(inst) >= 0
    ]

    if posiciones:
        posicion, _ = min(posiciones, key=lambda x: x[0])
        etiqueta = entrada[:posicion].strip(" ,;")
    else:
        etiqueta = entrada.split(",", 1)[0].strip()

    valido = nombres_compatibles(autor, etiqueta)

    # Respaldo conservador por apellido.
    if not valido:
        autor_limpio = quitar_scopus_id(autor)
        apellido = (
            autor_limpio.split(",", 1)[0]
            if "," in autor_limpio
            else autor_limpio.split()[-1]
        )
        apellido = normalizar_texto(apellido)
        etiqueta_norm = normalizar_texto(etiqueta)

        valido = bool(
            apellido
            and (
                etiqueta_norm == apellido
                or etiqueta_norm.startswith(apellido + " ")
            )
        )

    return encontradas, valido


def corresponding_wos(texto):
    resultado = []

    for segmento in separar_punto_coma(texto):
        m = re.match(
            r"^(.*?)\s*\(corresponding author\)\s*,\s*(.*)$",
            segmento,
            re.I
        )

        if m:
            resultado.append((m.group(1).strip(), m.group(2).strip()))

    return resultado


def es_afiliacion_unam(afiliacion):
    """
    Solo señales suficientemente específicas.
    Las variantes ambiguas se resuelven con la tutora o pasan a revisión.
    """
    texto = normalizar_texto(afiliacion)

    patrones = [
        r"\bunam\b",
        r"\bu n a m\b",
        r"nacional autonoma de mexico",
        r"national autonomous university of mexico",
        r"\biimas\b",
        r"instituto de investigaciones en matematicas aplicadas y en sistemas",
        r"institute for applied mathematics and systems research",
        r"posgrado en ciencia e ingenieria (?:de|en) la computacion",
        r"posgrado en ciencias de la computacion",
        r"\bdgtic\b",
        r"direccion general de computo y de tecnologias de informacion y comunicacion",
        r"ciudad universitaria.*coyoacan",
    ]

    return any(re.search(patron, texto) for patron in patrones)

## 5. CARGAR DATOS

In [ ]:
hash_antes = sha256(archivo)

df = pd.read_csv(
    archivo,
    dtype=str,
    keep_default_na=False,
    encoding="utf-8-sig"
)

# El CSV actual puede traer columnas Unnamed totalmente vacías.
columnas_unnamed = [c for c in df.columns if c.startswith("Unnamed")]

for c in columnas_unnamed:
    if df[c].str.strip().ne("").any():
        raise ValueError(f"La columna {c} contiene datos y no puede eliminarse.")

if columnas_unnamed:
    df = df.drop(columns=columnas_unnamed)

faltantes = [c for c in columnas if c not in df.columns]
extras = [c for c in df.columns if c not in columnas]

if faltantes:
    raise ValueError(f"Faltan columnas canónicas: {faltantes}")

if extras:
    raise ValueError(f"Existen columnas inesperadas: {extras}")

df = df[columnas].copy()

# Identificador estrictamente temporal. Nunca se guarda en la salida.
df["_row_id"] = range(len(df))


if os.path.exists(archivo_tutora_xlsx):
    tutora = pd.read_excel(
        archivo_tutora_xlsx,
        dtype=str,
        keep_default_na=False
    )
elif os.path.exists(archivo_tutora_csv):
    tutora = pd.read_csv(
        archivo_tutora_csv,
        dtype=str,
        keep_default_na=False,
        encoding="utf-8-sig"
    )
else:
    raise FileNotFoundError(
        "No se encontró UNAM_Completo_Corregido en ../00_control/"
    )

columnas_tutora = ["Titulo", "Año", "Autor_norm", "Afiliacion1", "Afiliacion2", "Doi"]
faltantes_tutora = [c for c in columnas_tutora if c not in tutora.columns]

if faltantes_tutora:
    raise ValueError(
        f"Faltan columnas en el archivo de la tutora: {faltantes_tutora}"
    )

print("Filas de entrada:", len(df))
print("Columnas canónicas:", len(columnas))
print("Filas archivo tutora:", len(tutora))

## 6. CASOS GRANDES: >10 AUTORES / AFILIACIONES / REFERENCIAS

In [ ]:
def contar_estructura(fila):
    fuente = fila["Fuente_origen"].strip()
    autores = separar_punto_coma(fila["Autor_norm"])

    catalogo_numerado = mapa_afiliaciones_numeradas(
        fila["Afiliacion1"],
        fila["Afiliacion2"]
    )
    hay_referencias = any(referencias_autor(autor) for autor in autores)

    if fuente == "Scopus":
        n_afiliaciones = len(separar_punto_coma(fila["Afiliacion2"]))
        n_referencias = 0

    elif fuente == "IEEE" and not (catalogo_numerado and hay_referencias):
        n_afiliaciones = len(separar_punto_coma(fila["Afiliacion1"]))
        n_referencias = 0

    elif catalogo_numerado and hay_referencias:
        n_afiliaciones = len(catalogo_numerado)
        n_referencias = len({
            ref
            for autor in autores
            for ref in referencias_autor(autor)
        })

    else:
        # En estructuras no numeradas el número de autores es la señal principal.
        n_afiliaciones = sum(
            bool(decodificar_html(fila[c]))
            for c in ("Afiliacion1", "Afiliacion2")
        )
        n_referencias = 0

    return pd.Series({
        "_n_autores": len(autores),
        "_n_afiliaciones": n_afiliaciones,
        "_n_referencias": n_referencias,
    })


df = pd.concat([df, df.apply(contar_estructura, axis=1)], axis=1)

df["_caso_grande"] = (
    df[["_n_autores", "_n_afiliaciones", "_n_referencias"]]
    .gt(10)
    .any(axis=1)
)

print(
    "Casos grandes enviados directamente a revisión:",
    int(df["_caso_grande"].sum())
)

## 7. PRIMERA PASADA: SEPARACIÓN AUTOR-AFILIACIÓN

In [ ]:
def separar_fila(fila):
    if fila["_caso_grande"]:
        return []

    fuente = fila["Fuente_origen"].strip()
    autores = separar_punto_coma(fila["Autor_norm"])

    base = {
        "row_id": fila["_row_id"],
        "Fuente_origen": fila["Fuente_origen"],
        "indice": fila["indice"],
        "Titulo": fila["Titulo"],
        "Año": fila["Año"],
        "Doi": fila["Doi"],
    }

    relaciones = []

    catalogo_numerado = mapa_afiliaciones_numeradas(
        fila["Afiliacion1"],
        fila["Afiliacion2"]
    )
    hay_referencias = any(referencias_autor(autor) for autor in autores)

    # A. Estructura numerada. Tiene prioridad sobre el nombre de la editorial.
    #    Esto permite trabajar también con registros ORCID.
    if fuente != "Scopus" and catalogo_numerado and hay_referencias:
        for autor in autores:
            refs = referencias_autor(autor)
            conocidas = [catalogo_numerado[r] for r in refs if r in catalogo_numerado]
            faltantes = [r for r in refs if r not in catalogo_numerado]

            relaciones.append({
                **base,
                "Autor": quitar_referencias_autor(autor),
                "Afiliaciones": unicos_no_vacios(conocidas),
                "Estado_separacion": (
                    "COMPLETO" if refs and not faltantes else "PENDIENTE"
                ),
                "Evidencia": "referencias numéricas",
            })

        return relaciones

    # B. EV debe tener referencias numéricas.
    if fuente == "EV":
        for autor in autores:
            relaciones.append({
                **base,
                "Autor": quitar_referencias_autor(autor),
                "Afiliaciones": [],
                "Estado_separacion": "PENDIENTE",
                "Evidencia": "EV sin estructura numérica resoluble",
            })
        return relaciones

    # C. IEEE: listas paralelas autor[i] <-> afiliacion[i].
    if fuente == "IEEE":
        afiliaciones = separar_punto_coma(fila["Afiliacion1"])
        valido = len(autores) > 0 and len(autores) == len(afiliaciones)

        for i, autor in enumerate(autores):
            afiliacion = afiliaciones[i] if valido else ""

            relaciones.append({
                **base,
                "Autor": decodificar_html(autor),
                "Afiliaciones": [afiliacion] if afiliacion else [],
                "Estado_separacion": "COMPLETO" if afiliacion else "PENDIENTE",
                "Evidencia": (
                    "IEEE listas paralelas"
                    if valido
                    else "IEEE número de autores y afiliaciones distinto"
                ),
            })

        return relaciones

    # D. Scopus: Autor_norm[i] <-> Afiliacion2[i], validado nominalmente.
    if fuente == "Scopus":
        entradas = separar_punto_coma(fila["Afiliacion2"])
        catalogo = separar_punto_coma(fila["Afiliacion1"])
        misma_longitud = len(autores) > 0 and len(autores) == len(entradas)

        for i, autor_original in enumerate(autores):
            autor = quitar_scopus_id(autor_original)
            afiliaciones = []
            valido = False

            if misma_longitud:
                afiliaciones, valido = afiliaciones_scopus(
                    autor_original,
                    entradas[i],
                    catalogo
                )

            relaciones.append({
                **base,
                "Autor": autor,
                "Afiliaciones": unicos_no_vacios(afiliaciones),
                "Estado_separacion": (
                    "COMPLETO" if valido and afiliaciones else "PENDIENTE"
                ),
                "Evidencia": (
                    "Scopus Afiliacion2 + catálogo Afiliacion1"
                    if valido and afiliaciones
                    else "Scopus no resoluble"
                ),
            })

        return relaciones

    # E. WoS: solo corresponding authors son relaciones directas.
    if fuente == "WoS":
        correspondientes = corresponding_wos(fila["Afiliacion2"])

        for autor in autores:
            afiliaciones = [
                afiliacion
                for etiqueta, afiliacion in correspondientes
                if nombres_compatibles(autor, etiqueta)
            ]
            afiliaciones = unicos_no_vacios(afiliaciones)

            relaciones.append({
                **base,
                "Autor": decodificar_html(autor),
                "Afiliaciones": afiliaciones,
                "Estado_separacion": "COMPLETO" if afiliaciones else "PENDIENTE",
                "Evidencia": (
                    "WoS corresponding author" if afiliaciones else "WoS pendiente"
                ),
            })

        return relaciones

    # F. Registro de un solo autor:
    #    ACM / ProQuest / ScienceDirect originales y editoriales ORCID.
    if len(autores) == 1:
        if catalogo_numerado:
            afiliaciones = list(catalogo_numerado.values())
            evidencia = "único autor + catálogo numerado"
        else:
            afiliaciones = [fila["Afiliacion1"], fila["Afiliacion2"]]
            evidencia = "registro individual"

        afiliaciones = unicos_no_vacios(afiliaciones)

        relaciones.append({
            **base,
            "Autor": decodificar_html(autores[0]),
            "Afiliaciones": afiliaciones,
            "Estado_separacion": "COMPLETO" if afiliaciones else "PENDIENTE",
            "Evidencia": evidencia,
        })

        return relaciones

    # G. Multiautor sin relación explícita.
    #    Se intentará resolver con otra representación del mismo artículo.
    for autor in autores:
        relaciones.append({
            **base,
            "Autor": decodificar_html(autor),
            "Afiliaciones": [],
            "Estado_separacion": "PENDIENTE",
            "Evidencia": "multiautor sin relación explícita",
        })

    return relaciones


registros = []

for _, fila in df.iterrows():
    registros.extend(separar_fila(fila))

relaciones = pd.DataFrame(registros)

print("Apariciones de autor extraídas:", len(relaciones))
print(relaciones["Estado_separacion"].value_counts())

## 8. SEGUNDA PASADA: MISMA PUBLICACIÓN EN OTRA FUENTE

In [ ]:
prioridad_evidencia = {
    "Scopus Afiliacion2 + catálogo Afiliacion1": 1,
    "referencias numéricas": 1,
    "IEEE listas paralelas": 2,
    "registro individual": 2,
    "único autor + catálogo numerado": 2,
    "WoS corresponding author": 2,
}

por_doi = defaultdict(list)
por_titulo_anio = defaultdict(list)

for i, r in relaciones[relaciones["Estado_separacion"].eq("COMPLETO")].iterrows():
    doi = normalizar_doi(r["Doi"])

    if doi:
        por_doi[doi].append(i)

    clave = (normalizar_titulo(r["Titulo"]), str(r["Año"]).strip())

    if clave[0]:
        por_titulo_anio[clave].append(i)


recuperadas_contexto = 0

for i, r in relaciones[~relaciones["Estado_separacion"].eq("COMPLETO")].iterrows():
    doi = normalizar_doi(r["Doi"])

    if doi:
        candidatos_base = por_doi.get(doi, [])
    else:
        clave = (normalizar_titulo(r["Titulo"]), str(r["Año"]).strip())
        candidatos_base = por_titulo_anio.get(clave, [])

    candidatos = [
        j
        for j in candidatos_base
        if relaciones.at[j, "row_id"] != r["row_id"]
        and nombres_compatibles(r["Autor"], relaciones.at[j, "Autor"])
    ]

    if not candidatos:
        continue

    mejor_prioridad = min(
        prioridad_evidencia.get(relaciones.at[j, "Evidencia"], 5)
        for j in candidatos
    )

    mejores = [
        j
        for j in candidatos
        if prioridad_evidencia.get(relaciones.at[j, "Evidencia"], 5)
        == mejor_prioridad
    ]

    afiliaciones = unicos_no_vacios([
        afiliacion
        for j in mejores
        for afiliacion in relaciones.at[j, "Afiliaciones"]
    ])

    if afiliaciones:
        relaciones.at[i, "Afiliaciones"] = afiliaciones
        relaciones.at[i, "Estado_separacion"] = "COMPLETO"
        relaciones.at[i, "Evidencia"] = "misma publicación en otra fuente"
        recuperadas_contexto += 1

print(
    "Relaciones recuperadas mediante otra representación:",
    recuperadas_contexto
)

## 9. TERCERA PASADA: ARCHIVO DE LA TUTORA

In [ ]:
tutora = tutora.copy()
tutora["_doi_norm"] = tutora["Doi"].map(normalizar_doi)
tutora["_titulo_norm"] = tutora["Titulo"].map(normalizar_titulo)
tutora["_anio_norm"] = tutora["Año"].astype(str).str.strip()

tutora_por_doi = defaultdict(list)
tutora_por_titulo_anio = defaultdict(list)

for i, fila in tutora.iterrows():
    if fila["_doi_norm"]:
        tutora_por_doi[fila["_doi_norm"]].append(i)

    if fila["_titulo_norm"]:
        tutora_por_titulo_anio[
            (fila["_titulo_norm"], fila["_anio_norm"])
        ].append(i)


def coincidencias_tutora(relacion):
    doi = normalizar_doi(relacion["Doi"])

    if doi:
        candidatos = tutora_por_doi.get(doi, [])
    else:
        clave = (
            normalizar_titulo(relacion["Titulo"]),
            str(relacion["Año"]).strip()
        )
        candidatos = tutora_por_titulo_anio.get(clave, [])

    return [
        j
        for j in candidatos
        if nombres_compatibles(
            relacion["Autor"],
            tutora.at[j, "Autor_norm"]
        )
    ]


relaciones["Coincide_tutora"] = False
recuperadas_tutora = 0

for i, r in relaciones.iterrows():
    coincidencias = coincidencias_tutora(r)

    if not coincidencias:
        continue

    relaciones.at[i, "Coincide_tutora"] = True

    if r["Estado_separacion"] != "COMPLETO":
        afiliaciones = unicos_no_vacios([
            afiliacion
            for j in coincidencias
            for afiliacion in (
                tutora.at[j, "Afiliacion1"],
                tutora.at[j, "Afiliacion2"]
            )
        ])

        if afiliaciones:
            relaciones.at[i, "Afiliaciones"] = afiliaciones
            relaciones.at[i, "Estado_separacion"] = "COMPLETO"
            relaciones.at[i, "Evidencia"] = "archivo tutora: publicación + autor"
            recuperadas_tutora += 1

print("Relaciones recuperadas mediante la tutora:", recuperadas_tutora)

## 10. FILAS QUE VAN A REVISIÓN MANUAL

In [ ]:
motivos_revision = defaultdict(list)

# Casos grandes.
for row_id in df.loc[df["_caso_grande"], "_row_id"]:
    motivos_revision[row_id].append(">10 autores/afiliaciones/referencias")

# Algún autor sigue sin relación segura.
filas_pendientes = set(
    relaciones.loc[
        ~relaciones["Estado_separacion"].eq("COMPLETO"),
        "row_id"
    ]
)

for row_id in filas_pendientes:
    motivos_revision[row_id].append("separación autor-afiliación no resuelta")

# El modelo final tiene máximo dos afiliaciones por autor.
# Si se recuperaron más de dos, no se truncan automáticamente.
filas_mas_dos = set(
    relaciones.loc[
        relaciones["Afiliaciones"].map(lambda x: len(x) > 2),
        "row_id"
    ]
)

for row_id in filas_mas_dos:
    motivos_revision[row_id].append("autor con más de dos afiliaciones")

filas_revision = set(motivos_revision.keys())

diagnostico_revision = pd.DataFrame([
    {
        "_row_id": row_id,
        "Motivo_revision": " | ".join(motivos)
    }
    for row_id, motivos in sorted(motivos_revision.items())
])

print("Filas originales para revisión manual:", len(filas_revision))

if len(diagnostico_revision) > 0:
    print()
    print("Motivos de revisión:")
    print(diagnostico_revision["Motivo_revision"].value_counts())

## 11. CLASIFICAR UNAM / EXTERNO

In [ ]:
relaciones_automaticas = relaciones[
    ~relaciones["row_id"].isin(filas_revision)
].copy()


def clasificar_unam(fila):
    if any(es_afiliacion_unam(a) for a in fila["Afiliaciones"]):
        return "UNAM"

    # El archivo de la tutora se acepta como evidencia autoritativa
    # solo cuando coincide la misma publicación y el mismo autor.
    if bool(fila["Coincide_tutora"]):
        return "UNAM"

    return "EXTERNO"


relaciones_automaticas["Estado_UNAM"] = relaciones_automaticas.apply(
    clasificar_unam,
    axis=1
)

print()
print("Clasificación automática publicación + autor:")
print(relaciones_automaticas["Estado_UNAM"].value_counts())

## 12. CONSTRUIR SALIDA AUTOMÁTICA

In [ ]:
relaciones_unam = relaciones_automaticas[
    relaciones_automaticas["Estado_UNAM"].eq("UNAM")
].copy()

salida_trabajo = []

for _, r in relaciones_unam.iterrows():
    original = df.loc[
        df["_row_id"].eq(r["row_id"]),
        columnas
    ].iloc[0].copy()

    afiliaciones = unicos_no_vacios(r["Afiliaciones"])

    if len(afiliaciones) > 2:
        raise RuntimeError(
            "Llegó a la salida automática un autor con más de dos afiliaciones."
        )

    original["Autor_norm"] = r["Autor"]
    original["Afiliacion1"] = afiliaciones[0] if len(afiliaciones) >= 1 else ""
    original["Afiliacion2"] = afiliaciones[1] if len(afiliaciones) >= 2 else ""

    salida_trabajo.append((r["row_id"], original))


if salida_trabajo:
    autores_unam_automatico = pd.DataFrame(
        [fila for _, fila in salida_trabajo]
    )
    autores_unam_automatico["_row_id"] = [
        row_id for row_id, _ in salida_trabajo
    ]
else:
    autores_unam_automatico = pd.DataFrame(
        columns=columnas + ["_row_id"]
    )

## 13. ÚNICO ARCHIVO DE REVISIÓN MANUAL

In [ ]:
casos_revision = df[
    df["_row_id"].isin(filas_revision)
][columnas].copy()

autores_unam_automatico[columnas].to_csv(
    salida_automatica,
    index=False,
    encoding="utf-8-sig"
)

casos_revision.to_csv(
    revision_manual,
    index=False,
    encoding="utf-8-sig"
)

## 14. VALIDACIONES

In [ ]:
automatico = autores_unam_automatico.copy()

if list(automatico[columnas].columns) != columnas:
    raise RuntimeError("La salida no conserva las 15 columnas canónicas.")

if any(c.startswith("Unnamed") for c in automatico.columns):
    raise RuntimeError("La salida contiene columnas Unnamed.")

if automatico["Autor_norm"].str.strip().eq("").any():
    raise RuntimeError("La salida automática contiene autores vacíos.")

if automatico["Autor_norm"].str.contains(";", regex=False).any():
    raise RuntimeError("Hay más de un autor dentro de Autor_norm.")

af1 = automatico["Afiliacion1"].map(normalizar_texto)
af2 = automatico["Afiliacion2"].map(normalizar_texto)

duplicada_afiliacion = af1.ne("") & af2.ne("") & af1.eq(af2)

if duplicada_afiliacion.any():
    raise RuntimeError("Hay filas con Afiliacion1 y Afiliacion2 idénticas.")

columnas_inmutables = [
    "Fuente_origen", "indice", "Titulo", "Año",
    "ISBN", "ISSN", "Doi", "URL",
    "Area", "SubArea", "Keywords", "Abstract"
]

for _, fila_salida in automatico.iterrows():
    original = df.loc[
        df["_row_id"].eq(fila_salida["_row_id"])
    ].iloc[0]

    for columna in columnas_inmutables:
        if fila_salida[columna] != original[columna]:
            raise RuntimeError(
                f"Se modificó una columna no permitida: {columna}"
            )

if automatico["_row_id"].isin(filas_revision).any():
    raise RuntimeError(
        "La salida automática incluye una fila enviada a revisión."
    )

## 15. INCORPORACIÓN OPCIONAL DE LA REVISIÓN MANUAL

In [ ]:
final_generado = False

if os.path.exists(revision_resuelta):
    manual = pd.read_csv(
        revision_resuelta,
        dtype=str,
        keep_default_na=False,
        encoding="utf-8-sig"
    )

    unnamed = [c for c in manual.columns if c.startswith("Unnamed")]

    for c in unnamed:
        if manual[c].str.strip().ne("").any():
            raise ValueError(f"{c} del archivo manual contiene datos.")

    if unnamed:
        manual = manual.drop(columns=unnamed)

    if list(manual.columns) != columnas:
        raise ValueError(
            "casos_revision_resueltos.csv debe tener exactamente "
            "las 15 columnas canónicas."
        )

    if manual["Autor_norm"].str.strip().eq("").any():
        raise ValueError("La revisión resuelta contiene autores vacíos.")

    if manual["Autor_norm"].str.contains(";", regex=False).any():
        raise ValueError(
            "Cada fila de la revisión resuelta debe contener un solo autor."
        )

    manual_af1 = manual["Afiliacion1"].map(normalizar_texto)
    manual_af2 = manual["Afiliacion2"].map(normalizar_texto)

    if (
        manual_af1.ne("")
        & manual_af2.ne("")
        & manual_af1.eq(manual_af2)
    ).any():
        raise ValueError(
            "La revisión resuelta contiene Afiliacion1 == Afiliacion2."
        )

    # Una fila manual debe corresponder a una publicación enviada a revisión.
    def clave_publicacion(fila):
        return (
            normalizar_texto(fila["Fuente_origen"]),
            str(fila["indice"]).strip(),
            normalizar_titulo(fila["Titulo"]),
            str(fila["Año"]).strip(),
            normalizar_doi(fila["Doi"]),
        )

    claves_revision = {
        clave_publicacion(fila)
        for _, fila in casos_revision.iterrows()
    }

    claves_manual = {
        clave_publicacion(fila)
        for _, fila in manual.iterrows()
    }

    if not claves_manual.issubset(claves_revision):
        raise ValueError(
            "La revisión resuelta contiene publicaciones "
            "que no estaban en casos_revision_manual.csv."
        )

    final = pd.concat(
        [
            automatico[columnas],
            manual[columnas]
        ],
        ignore_index=True
    )

    final.to_csv(
        salida_final,
        index=False,
        encoding="utf-8-sig"
    )

    final_generado = True

else:
    # Evita confundir un resultado viejo con la ejecución actual.
    if os.path.exists(salida_final):
        os.remove(salida_final)

## 16. RESUMEN FINAL

In [ ]:
hash_despues = sha256(archivo)

if hash_antes != hash_despues:
    raise RuntimeError("Canonico_Union_Trabajo.csv fue modificado.")


print()
print("RESUMEN FINAL")
print("=============")
print("Filas de entrada:", len(df))
print(
    "Filas originales con procesamiento automático:",
    len(df) - len(filas_revision)
)
print("Filas originales enviadas a revisión:", len(filas_revision))
print("  Casos grandes (>10):", int(df["_caso_grande"].sum()))
print(
    "  Con relación autor-afiliación pendiente:",
    len(filas_pendientes)
)
print("  Con autor de >2 afiliaciones:", len(filas_mas_dos))
print(
    "Apariciones de autor procesadas automáticamente:",
    len(relaciones_automaticas)
)
print(
    "Autores externos eliminados automáticamente:",
    int(relaciones_automaticas["Estado_UNAM"].eq("EXTERNO").sum())
)
print("Filas UNAM automáticas:", len(autores_unam_automatico))
print(
    "Autores UNAM distintos automáticos:",
    autores_unam_automatico["Autor_norm"].nunique()
)
print(
    "Afiliacion1 vacía en salida automática:",
    int(autores_unam_automatico["Afiliacion1"].str.strip().eq("").sum())
)
print(
    "Afiliacion2 vacía en salida automática:",
    int(autores_unam_automatico["Afiliacion2"].str.strip().eq("").sum())
)
print("Afiliacion1 == Afiliacion2:", int(duplicada_afiliacion.sum()))
print("Columnas salida automática:", len(columnas))
print("Archivo original intacto:", hash_antes == hash_despues)
print()
print("Salida automática:", salida_automatica)
print("Revisión manual:", revision_manual)

if final_generado:
    print("Salida final:", salida_final)
else:
    print()
    print(
        "Todavía no se generó autores_unam_separados.csv porque "
        "no existe casos_revision_resueltos.csv."
    )
    print(
        "Resuelve manualmente casos_revision_manual.csv, guarda "
        "solo las filas finales de autores UNAM como "
        "casos_revision_resueltos.csv con las mismas 15 columnas "
        "y vuelve a ejecutar el notebook."
    )